# Q2, Q3, Q4 — Linear, Polynomial, Lasso & Ridge Regression
**Dataset:** Housing.csv

**For all questions:**
- Split into train/test using K-Fold Cross Validation
- Report MSE and R² for each fold
- Report Average MSE and R²

---
- **Q2:** Simple Linear Regression + Multiple Linear Regression
- **Q3:** Polynomial Regression
- **Q4:** Lasso and Ridge Regression for Multiple Polynomial Regression

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Import all required libraries
# ─────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, r2_score

print('All libraries imported successfully!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Load and Explore the Housing Dataset
# ─────────────────────────────────────────────

# Load the CSV file — make sure Housing.csv is in the same folder as this notebook
df = pd.read_csv('Housing.csv')

print('Shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Preprocessing: Handle Categorical Columns
#
# Housing.csv has some yes/no columns like 'mainroad',
# 'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
# 'prefarea', 'furnishingstatus'
#
# Machine Learning models need numbers, not strings.
# pd.get_dummies() converts yes/no → 1/0
# drop_first=True avoids the "dummy variable trap"
# (if a column has only yes/no, one column is enough)
# ─────────────────────────────────────────────

# Check which columns are non-numeric (object type)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns found:', cat_cols)

# Convert all categorical columns to numeric using one-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True)

print('\nShape after encoding:', df_encoded.shape)
print('\nAll columns after encoding:')
print(df_encoded.columns.tolist())
df_encoded.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Prepare X and y
#
# Target column (what we want to predict): 'price'
# Feature columns (inputs to the model): everything else
# ─────────────────────────────────────────────

# y = house price (what we predict)
y = df_encoded['price'].values

# X = all other columns (features)
X_all = df_encoded.drop('price', axis=1).values

# For Simple Linear Regression (Q2 part 1),
# we also need a single-feature version.
# 'area' is the most logical single predictor of price.
# Using column index 0 which corresponds to 'area'.
X_simple = df_encoded[['area']].values   # shape (n, 1) — 2D needed by sklearn

print('X_all shape  (multiple features):', X_all.shape)
print('X_simple shape (area only)       :', X_simple.shape)
print('y shape                          :', y.shape)

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Normalize X
#
# StandardScaler makes mean=0 and std=1 for each column.
# Important for regression so large-valued features
# (like area in sqft) don't dominate small ones (like bedrooms).
# ─────────────────────────────────────────────

scaler_all    = StandardScaler()
scaler_simple = StandardScaler()

X_all_scaled    = scaler_all.fit_transform(X_all)
X_simple_scaled = scaler_simple.fit_transform(X_simple)

print('X_all normalized    — mean:', X_all_scaled.mean(axis=0).round(3))
print('X_simple normalized — mean:', X_simple_scaled.mean(axis=0).round(3))

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Helper function to run K-Fold and report metrics
#
# Instead of rewriting the same loop 5 times,
# we define a reusable function.
# Pass in X, y, and any sklearn model object.
# It runs 5-fold CV and prints MSE + R² for each fold.
# ─────────────────────────────────────────────

def run_kfold(X, y, model, model_name='Model'):
    """
    Runs 5-Fold Cross Validation on given model.
    Prints MSE and R2 for each fold and averages.
    Returns (avg_mse, avg_r2)
    """
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    mse_list = []
    r2_list  = []

    print(f'\n{"-"*55}')
    print(f'  {model_name}')
    print(f'{"-"*55}')

    for fold_num, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        # Split data
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Train
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        # Metrics
        # MSE  = average of (actual - predicted)^2
        #        lower is better (0 = perfect)
        # R²   = how much variance in y the model explains
        #        1.0 = perfect, 0 = model predicts just the mean, <0 = worse than mean
        mse = mean_squared_error(y_test, y_pred)
        r2  = r2_score(y_test, y_pred)

        mse_list.append(mse)
        r2_list.append(r2)

        print(f'  Fold {fold_num} — MSE: {mse:>15.2f}   R²: {r2:.4f}')

    avg_mse = np.mean(mse_list)
    avg_r2  = np.mean(r2_list)

    print(f'  {"─"*50}')
    print(f'  AVG MSE : {avg_mse:>15.2f}')
    print(f'  AVG R²  : {avg_r2:.4f}')

    return avg_mse, avg_r2

print('Helper function defined.')

---
## Q2 — Simple Linear Regression and Multiple Linear Regression

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Q2 Part A: Simple Linear Regression
#
# Simple = only ONE input feature (area)
# Model learns: price = w1 * area + b
# ─────────────────────────────────────────────

simple_lr = LinearRegression()
run_kfold(X_simple_scaled, y, simple_lr, 'Q2 — Simple Linear Regression (area only)')

# Train on full data just to show the learned equation
simple_lr.fit(X_simple_scaled, y)
print(f'\n  Learned equation: price = {simple_lr.coef_[0]:.2f} * area + {simple_lr.intercept_:.2f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Q2 Part A: Scatter plot with regression line
# ─────────────────────────────────────────────

plt.figure(figsize=(8, 5))

# Scatter: actual data points
plt.scatter(X_simple_scaled, y, color='steelblue', alpha=0.5, label='Actual data')

# Line: model predictions across sorted range
x_line = np.linspace(X_simple_scaled.min(), X_simple_scaled.max(), 200).reshape(-1, 1)
y_line = simple_lr.predict(x_line)
plt.plot(x_line, y_line, color='red', lw=2, label='Regression line')

plt.xlabel('Area (normalized)')
plt.ylabel('Price')
plt.title('Q2 — Simple Linear Regression: Area vs Price')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Q2 Part B: Multiple Linear Regression
#
# Multiple = ALL features used together
# Model learns: price = w1*area + w2*bedrooms + w3*bathrooms + ... + b
# ─────────────────────────────────────────────

multi_lr = LinearRegression()
run_kfold(X_all_scaled, y, multi_lr, 'Q2 — Multiple Linear Regression (all features)')

# Show coefficients for each feature
multi_lr.fit(X_all_scaled, y)
feature_names = df_encoded.drop('price', axis=1).columns.tolist()
print('\n  Feature coefficients (weights):')
for fname, coef in zip(feature_names, multi_lr.coef_):
    print(f'    {fname:30s}: {coef:.2f}')
print(f'  Intercept (b): {multi_lr.intercept_:.2f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Q2: Actual vs Predicted plot for Multiple LR
#
# A good model: points should lie close to the diagonal line
# (predicted = actual)
# ─────────────────────────────────────────────

y_pred_full = multi_lr.predict(X_all_scaled)

plt.figure(figsize=(7, 6))
plt.scatter(y, y_pred_full, alpha=0.5, color='steelblue')

# Perfect prediction line
min_val = min(y.min(), y_pred_full.min())
max_val = max(y.max(), y_pred_full.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')

plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Q2 — Multiple Linear Regression: Actual vs Predicted')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Q3 — Polynomial Regression

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Q3: Polynomial Regression (degree=2)
#
# Polynomial regression fits curves, not just straight lines.
# It does this by creating NEW features from the existing ones:
#   Original features: [area, bedrooms]
#   After degree=2:    [1, area, bedrooms, area², area*bedrooms, bedrooms²]
# Then normal LinearRegression is applied on these expanded features.
#
# IMPORTANT: PolynomialFeatures must be applied INSIDE the fold loop
# (fit on train, transform both train and test)
# so test data doesn't influence the polynomial terms
# ─────────────────────────────────────────────

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_list_poly = []
r2_list_poly  = []

print('─'*55)
print('  Q3 — Polynomial Regression (degree=2, all features)')
print('─'*55)

for fold_num, (train_idx, test_idx) in enumerate(kf.split(X_all_scaled), start=1):

    X_train, X_test = X_all_scaled[train_idx], X_all_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Step 1: Create polynomial features
    # fit_transform on train: learns feature names, creates squared/cross terms
    # transform on test: applies same transformation (NO fitting on test)
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly  = poly.transform(X_test)

    # Step 2: Train LinearRegression on the expanded features
    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    # Step 3: Predict and evaluate
    y_pred = model.predict(X_test_poly)

    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    mse_list_poly.append(mse)
    r2_list_poly.append(r2)

    print(f'  Fold {fold_num} — MSE: {mse:>15.2f}   R²: {r2:.4f}')

print(f'  {"─"*50}')
print(f'  AVG MSE : {np.mean(mse_list_poly):>15.2f}')
print(f'  AVG R²  : {np.mean(r2_list_poly):.4f}')

# Show how many features were created
poly_demo = PolynomialFeatures(degree=2, include_bias=False)
X_poly_demo = poly_demo.fit_transform(X_all_scaled)
print(f'\n  Original features : {X_all_scaled.shape[1]}')
print(f'  After degree=2    : {X_poly_demo.shape[1]} (added squared + cross terms)')

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 — Q3: Compare degree=1 vs degree=2 vs degree=3
#
# Higher degree = more complex curve = fits training data better
# BUT may overfit (perform worse on test data)
# This plot shows how R² changes with degree
# ─────────────────────────────────────────────

degrees = [1, 2, 3]
avg_r2_by_degree = []

for deg in degrees:
    r2_folds = []
    for train_idx, test_idx in kf.split(X_all_scaled):
        X_train, X_test = X_all_scaled[train_idx], X_all_scaled[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        poly = PolynomialFeatures(degree=deg, include_bias=False)
        X_train_p = poly.fit_transform(X_train)
        X_test_p  = poly.transform(X_test)

        m = LinearRegression()
        m.fit(X_train_p, y_train)
        r2_folds.append(r2_score(y_test, m.predict(X_test_p)))

    avg_r2_by_degree.append(np.mean(r2_folds))
    print(f'  Degree {deg} → Avg R²: {np.mean(r2_folds):.4f}')

plt.figure(figsize=(7, 4))
plt.plot(degrees, avg_r2_by_degree, marker='o', color='steelblue', lw=2)
plt.xlabel('Polynomial Degree')
plt.ylabel('Average R² (5-Fold CV)')
plt.title('Q3 — Polynomial Regression: Degree vs R²')
plt.xticks(degrees)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Q4 — Lasso and Ridge Regression for Multiple Polynomial Regression

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — What are Lasso and Ridge?
#
# Problem with Polynomial Regression:
# degree=2 creates many features → model can overfit
# (learns noise in training data, performs badly on test)
#
# Lasso (L1) and Ridge (L2) add a PENALTY for large weights.
# This forces the model to keep weights small → less overfitting.
#
# Ridge (L2): penalty = alpha * sum(w²)
#   → shrinks all weights toward 0 but keeps all features
#
# Lasso (L1): penalty = alpha * sum(|w|)
#   → pushes some weights to exactly 0 (feature selection!)
#
# alpha = how strong the penalty is
#   alpha=0  → same as plain LinearRegression
#   alpha=∞  → all weights = 0 (useless model)
# ─────────────────────────────────────────────

print('Concept check:')
print('  Ridge  → keeps all features, shrinks weights')
print('  Lasso  → can zero out some features (automatic feature selection)')
print('  Both applied on top of Polynomial features (degree=2)')

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — Helper: K-Fold with Polynomial + Regularization
#
# Same as Cell 11 but accepts any regression model.
# Polynomial transformation happens INSIDE the loop.
# ─────────────────────────────────────────────

def run_kfold_poly(X, y, model, degree=2, model_name='Model'):
    """
    Runs 5-Fold CV with polynomial feature expansion + given model.
    Returns (avg_mse, avg_r2)
    """
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mse_list, r2_list = [], []

    print(f'\n{"-"*60}')
    print(f'  {model_name} (Polynomial degree={degree})')
    print(f'{"-"*60}')

    for fold_num, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Polynomial expansion — fit only on train!
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        X_train_p = poly.fit_transform(X_train)
        X_test_p  = poly.transform(X_test)

        # Train regularized model
        model.fit(X_train_p, y_train)
        y_pred = model.predict(X_test_p)

        mse = mean_squared_error(y_test, y_pred)
        r2  = r2_score(y_test, y_pred)
        mse_list.append(mse)
        r2_list.append(r2)

        print(f'  Fold {fold_num} — MSE: {mse:>15.2f}   R²: {r2:.4f}')

    avg_mse = np.mean(mse_list)
    avg_r2  = np.mean(r2_list)
    print(f'  {"─"*55}')
    print(f'  AVG MSE : {avg_mse:>15.2f}')
    print(f'  AVG R²  : {avg_r2:.4f}')

    return avg_mse, avg_r2

print('Helper defined.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 15 — Q4: Ridge Regression on Polynomial Features
#
# Ridge(alpha=1.0): alpha controls regularization strength
# Higher alpha → stronger penalty → smaller weights → simpler model
# ─────────────────────────────────────────────

ridge_model = Ridge(alpha=1.0)
ridge_mse, ridge_r2 = run_kfold_poly(
    X_all_scaled, y,
    ridge_model,
    degree=2,
    model_name='Q4 — Ridge Regression'
)

In [ ]:
# ─────────────────────────────────────────────
# CELL 16 — Q4: Lasso Regression on Polynomial Features
#
# Lasso(alpha=0.01): smaller alpha needed because
# Lasso's penalty is stronger per unit than Ridge.
# max_iter=10000 avoids convergence warnings.
# ─────────────────────────────────────────────

lasso_model = Lasso(alpha=0.01, max_iter=10000)
lasso_mse, lasso_r2 = run_kfold_poly(
    X_all_scaled, y,
    lasso_model,
    degree=2,
    model_name='Q4 — Lasso Regression'
)

In [ ]:
# ─────────────────────────────────────────────
# CELL 17 — Q4: How does alpha affect Lasso and Ridge?
#
# Too small alpha → no regularization → possible overfit
# Too large alpha → too much penalty → model underfits
# The sweet spot is somewhere in between.
# ─────────────────────────────────────────────

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

ridge_r2s = []
lasso_r2s = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for alpha in alphas:
    r2_ridge, r2_lasso = [], []

    for train_idx, test_idx in kf.split(X_all_scaled):
        X_train, X_test = X_all_scaled[train_idx], X_all_scaled[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_tr_p = poly.fit_transform(X_train)
        X_te_p = poly.transform(X_test)

        # Ridge
        rm = Ridge(alpha=alpha)
        rm.fit(X_tr_p, y_train)
        r2_ridge.append(r2_score(y_test, rm.predict(X_te_p)))

        # Lasso
        lm = Lasso(alpha=alpha, max_iter=10000)
        lm.fit(X_tr_p, y_train)
        r2_lasso.append(r2_score(y_test, lm.predict(X_te_p)))

    ridge_r2s.append(np.mean(r2_ridge))
    lasso_r2s.append(np.mean(r2_lasso))

plt.figure(figsize=(9, 5))
plt.plot(alphas, ridge_r2s, marker='o', label='Ridge', color='steelblue', lw=2)
plt.plot(alphas, lasso_r2s, marker='s', label='Lasso', color='coral',     lw=2)
plt.xscale('log')  # log scale because alpha range spans 0.001 to 100
plt.xlabel('Alpha (regularization strength) — log scale')
plt.ylabel('Average R² (5-Fold CV)')
plt.title('Q4 — Ridge vs Lasso: Effect of Alpha on R²')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Best Ridge alpha:', alphas[np.argmax(ridge_r2s)], f'→ R²={max(ridge_r2s):.4f}')
print('Best Lasso alpha:', alphas[np.argmax(lasso_r2s)], f'→ R²={max(lasso_r2s):.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 18 — Q4: Lasso Feature Selection
#
# Unique to Lasso: it zeros out unimportant feature weights.
# This tells us which polynomial features actually matter.
# ─────────────────────────────────────────────

# Train Lasso on full data to inspect weights
poly_full = PolynomialFeatures(degree=2, include_bias=False)
X_poly_full = poly_full.fit_transform(X_all_scaled)

lasso_full = Lasso(alpha=0.01, max_iter=10000)
lasso_full.fit(X_poly_full, y)

feature_names_poly = poly_full.get_feature_names_out(
    df_encoded.drop('price', axis=1).columns.tolist()
)
coefs = lasso_full.coef_

# Count how many were zeroed out
n_zero    = np.sum(coefs == 0)
n_nonzero = np.sum(coefs != 0)
print(f'Total polynomial features : {len(coefs)}')
print(f'Zeroed out by Lasso       : {n_zero}')
print(f'Kept (non-zero)           : {n_nonzero}')

# Plot top 15 non-zero coefficients
nonzero_mask = coefs != 0
nz_names = feature_names_poly[nonzero_mask]
nz_coefs = coefs[nonzero_mask]

# Sort by absolute value for plotting
sort_idx = np.argsort(np.abs(nz_coefs))[::-1][:15]

plt.figure(figsize=(9, 5))
colors = ['steelblue' if c > 0 else 'coral' for c in nz_coefs[sort_idx]]
plt.barh(nz_names[sort_idx][::-1], nz_coefs[sort_idx][::-1], color=colors[::-1])
plt.xlabel('Coefficient Value')
plt.title('Q4 — Lasso: Top Non-Zero Coefficients (Feature Selection)')
plt.axvline(x=0, color='black', lw=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 19 — Final Summary: All Models Compared
# ─────────────────────────────────────────────

print('=' * 62)
print('  FINAL COMPARISON — All Regression Models')
print('=' * 62)
print(f'{"Model":<40} {"Avg R²":>10}')
print('─' * 62)

# Re-run all models cleanly to collect final numbers
results = {}

# Simple LR
r2s = []
for tr, te in KFold(5, shuffle=True, random_state=42).split(X_simple_scaled):
    m = LinearRegression()
    m.fit(X_simple_scaled[tr], y[tr])
    r2s.append(r2_score(y[te], m.predict(X_simple_scaled[te])))
results['Q2 Simple Linear Regression'] = np.mean(r2s)

# Multiple LR
r2s = []
for tr, te in KFold(5, shuffle=True, random_state=42).split(X_all_scaled):
    m = LinearRegression()
    m.fit(X_all_scaled[tr], y[tr])
    r2s.append(r2_score(y[te], m.predict(X_all_scaled[te])))
results['Q2 Multiple Linear Regression'] = np.mean(r2s)

# Poly deg 2
r2s = []
for tr, te in KFold(5, shuffle=True, random_state=42).split(X_all_scaled):
    poly = PolynomialFeatures(2, include_bias=False)
    Xtr = poly.fit_transform(X_all_scaled[tr])
    Xte = poly.transform(X_all_scaled[te])
    m = LinearRegression(); m.fit(Xtr, y[tr])
    r2s.append(r2_score(y[te], m.predict(Xte)))
results['Q3 Polynomial Regression (deg=2)'] = np.mean(r2s)

# Ridge
r2s = []
for tr, te in KFold(5, shuffle=True, random_state=42).split(X_all_scaled):
    poly = PolynomialFeatures(2, include_bias=False)
    Xtr = poly.fit_transform(X_all_scaled[tr])
    Xte = poly.transform(X_all_scaled[te])
    m = Ridge(alpha=1.0); m.fit(Xtr, y[tr])
    r2s.append(r2_score(y[te], m.predict(Xte)))
results['Q4 Ridge Regression (alpha=1.0)'] = np.mean(r2s)

# Lasso
r2s = []
for tr, te in KFold(5, shuffle=True, random_state=42).split(X_all_scaled):
    poly = PolynomialFeatures(2, include_bias=False)
    Xtr = poly.fit_transform(X_all_scaled[tr])
    Xte = poly.transform(X_all_scaled[te])
    m = Lasso(alpha=0.01, max_iter=10000); m.fit(Xtr, y[tr])
    r2s.append(r2_score(y[te], m.predict(Xte)))
results['Q4 Lasso Regression (alpha=0.01)'] = np.mean(r2s)

for name, r2 in results.items():
    print(f'  {name:<40} {r2:>8.4f}')

best = max(results, key=results.get)
print('─' * 62)
print(f'  Best model: {best}')
print(f'  (R² closer to 1.0 = better fit)')

In [ ]:
# ─────────────────────────────────────────────
# CELL 20 — Summary Bar Chart
# ─────────────────────────────────────────────

plt.figure(figsize=(10, 5))
names  = list(results.keys())
values = list(results.values())
colors = ['#5B8DB8', '#5B8DB8', '#F4A46A', '#E06C75', '#98C379']

bars = plt.barh(names, values, color=colors, edgecolor='white')

for bar, val in zip(bars, values):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.xlabel('Average R² (5-Fold CV)')
plt.title('Q2 / Q3 / Q4 — Model Comparison by R²')
plt.xlim(0, max(values) + 0.1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()